In [2]:
%%bash
apt-get update && apt-get install -y openjdk-17-jre-headless

Get:1 https://packages.microsoft.com/repos/microsoft-debian-bookworm-prod bookworm InRelease [3618 B]
Get:2 https://packages.microsoft.com/repos/microsoft-debian-bookworm-prod bookworm/main amd64 Packages [153 kB]
Get:3 https://packages.microsoft.com/repos/microsoft-debian-bookworm-prod bookworm/main all Packages [573 B]
Get:4 http://deb.debian.org/debian bookworm InRelease [151 kB]
Get:5 http://deb.debian.org/debian bookworm-updates InRelease [55.4 kB]
Get:6 http://deb.debian.org/debian-security bookworm-security InRelease [48.0 kB]
Get:7 http://deb.debian.org/debian bookworm/main amd64 Packages [8792 kB]
Get:8 http://deb.debian.org/debian bookworm-updates/main amd64 Packages [6924 B]
Get:9 http://deb.debian.org/debian-security bookworm-security/main amd64 Packages [293 kB]
Fetched 9503 kB in 2s (5666 kB/s)
Reading package lists...
Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  alsa-topology-conf

debconf: delaying package configuration, since apt-utils is not installed


Fetched 51.8 MB in 1s (37.4 MB/s)
Selecting previously unselected package libdbus-1-3:amd64.
(Reading database ... 13036 files and directories currently installed.)
Preparing to unpack .../00-libdbus-1-3_1.14.10-1~deb12u1_amd64.deb ...
Unpacking libdbus-1-3:amd64 (1.14.10-1~deb12u1) ...
Selecting previously unselected package dbus-bin.
Preparing to unpack .../01-dbus-bin_1.14.10-1~deb12u1_amd64.deb ...
Unpacking dbus-bin (1.14.10-1~deb12u1) ...
Selecting previously unselected package dbus-session-bus-common.
Preparing to unpack .../02-dbus-session-bus-common_1.14.10-1~deb12u1_all.deb ...
Unpacking dbus-session-bus-common (1.14.10-1~deb12u1) ...
Selecting previously unselected package libapparmor1:amd64.
Preparing to unpack .../03-libapparmor1_3.0.8-3_amd64.deb ...
Unpacking libapparmor1:amd64 (3.0.8-3) ...
Selecting previously unselected package dbus-daemon.
Preparing to unpack .../04-dbus-daemon_1.14.10-1~deb12u1_amd64.deb ...
Unpacking dbus-daemon (1.14.10-1~deb12u1) ...
Selecting pr

In [1]:
import time
from minio import Minio
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, sha2, levenshtein, when, hour

# 1. INFRASTRUCTURE GUARD: Ensure MinIO buckets exist
def setup_storage():
    try:
        client = Minio("127.0.0.1:9000", 
                       access_key="admin", 
                       secret_key="sentinel_password", 
                       secure=False)
        for bucket in ["bronze", "silver"]:
            if not client.bucket_exists(bucket):
                print(f"🪣 Creating bucket: {bucket}")
                client.make_bucket(bucket)
        return True
    except Exception as e:
        print(f"❌ Storage Check Failed: {e}")
        return False

# 2. HIGH-PERFORMANCE SPARK INITIALIZATION
if setup_storage():
    if 'spark' in locals(): spark.stop()

    # We tune memory and shuffle partitions to prevent OutOfMemory errors
    spark = SparkSession.builder \
        .appName("Prism-Risk-Silver-Transformation") \
        .master("local[*]") \
        .config("spark.driver.memory", "2g") \
        .config("spark.executor.memory", "2g") \
        .config("spark.sql.shuffle.partitions", "50") \
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.0.0,org.apache.hadoop:hadoop-aws:3.3.4") \
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
        .config("spark.hadoop.fs.s3a.endpoint", "http://127.0.0.1:9000") \
        .config("spark.hadoop.fs.s3a.access.key", "admin") \
        .config("spark.hadoop.fs.s3a.secret.key", "sentinel_password") \
        .config("spark.hadoop.fs.s3a.path.style.access", "true") \
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
        .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
        .getOrCreate()

    spark.sparkContext.setLogLevel("ERROR")
    print("🚀 Silver Engine Online. Processing 10M rows...")

    try:
        # 3. READ & TRANSFORM
        # We use repartition(8) on the read to align with the CPU cores of a standard Codespace
        df_bronze = spark.read.format("delta").load("s3a://bronze/transactions_raw").repartition(8)
        
        TARGET_ENTITY = "NORTH_STAR_SHIPPING"
        
        df_silver = df_bronze \
            .withColumn("hashed_user_id", sha2(col("user_id").cast("string"), 256)) \
            .drop("user_id") \
            .withColumn("clock_hour", hour("timestamp")) \
            .withColumn("edit_distance", levenshtein(col("counterparty"), lit(TARGET_ENTITY))) \
            .withColumn("preliminary_risk_score", 
                when(col("edit_distance") < 8, 0.95)      # Sanctions Evasion Pattern
                .when(col("country") == "IRAN_PROXY", 0.70) # Geo-Risk Pattern
                .when((col("amount") > 9000) & (col("amount") < 10000), 0.40) # Smurfing Pattern
                .otherwise(0.01))

        # 4. MEMORY-SAFE WRITE
        print("📦 Persisting refined data to Silver Layer...")
        start_time = time.time()
        
        # Overwrite schema so dropped columns (e.g., user_id) are removed from Delta
        df_silver.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .save("s3a://silver/transactions_refined")
        
        print(f"🔥 SUCCESS: Silver Layer Ready in {round(time.time() - start_time, 2)}s")

        # 5. RISK AUDIT PREVIEW
        print("\n📊 SENTINEL DETECTION REPORT (Sample):")
        df_silver.filter(col("edit_distance") < 8) \
                 .select("counterparty", "edit_distance", "preliminary_risk_score") \
                 .limit(10).show()

    except Exception as e:
        print(f"❌ Transformation Failed: {e}")

else:
    print("🚨 Could not start Spark: Infrastructure Check Failed.")

26/02/08 12:10:12 WARN Utils: Your hostname, codespaces-ae75fe resolves to a loopback address: 127.0.0.1; using 10.0.2.255 instead (on interface eth0)
26/02/08 12:10:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-aac86216-9eb8-48b8-9360-c73783fe1362;1.0
	confs: [default]


:: loading settings :: url = jar:file:/opt/conda/envs/prism-risk/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-spark_2.12;3.0.0 in central
	found io.delta#delta-storage;3.0.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 188ms :: artifacts dl 7ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.0.0 from central in [default]
	io.delta#delta-storage;3.0.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	--

🚀 Silver Engine Online. Processing 10M rows...
📦 Persisting refined data to Silver Layer...


🔥 SUCCESS: Silver Layer Ready in 27.92s

📊 SENTINEL DETECTION REPORT (Sample):


+-------------------+-------------+----------------------+
|       counterparty|edit_distance|preliminary_risk_score|
+-------------------+-------------+----------------------+
|N0RTH_ST4R_SH1PP1NG|            4|                  0.95|
|NORTH_STAR_SHIPPING|            0|                  0.95|
|N0RTH_ST4R_SH1PP1NG|            4|                  0.95|
|NORTH_STAR_SHIPPING|            0|                  0.95|
|N0RTH_ST4R_SH1PP1NG|            4|                  0.95|
|NORTH_STAR_SHIPPING|            0|                  0.95|
|N0RTH_ST4R_SH1PP1NG|            4|                  0.95|
|NORTH_STAR_SHIPPING|            0|                  0.95|
|N0RTH_ST4R_SH1PP1NG|            4|                  0.95|
|NORTH_STAR_SHIPPING|            0|                  0.95|
+-------------------+-------------+----------------------+



In [2]:
from pyspark.sql.functions import col, count, avg, max, min

# 1. Initialize Validation Engine
print("🔍 Sentinel Silver Validation: Starting...")
df_silver = spark.read.format("delta").load("s3a://silver/transactions_refined")

# --- TEST 1: PII MASKING AUDIT ---
# We must ensure no raw 'user_id' exists and 'hashed_user_id' is present.
pii_check = "user_id" in df_silver.columns
has_hash = "hashed_user_id" in df_silver.columns

if not pii_check and has_hash:
    print("✅ PASS: PII Masking confirmed. Raw User IDs removed.")
else:
    print("❌ FAIL: Potential PII Leak detected!")

# --- TEST 2: FUZZY MATCHING ACCURACY ---
# Check if the 'Evasion Hunter' actually caught variations
evasions_count = df_silver.filter(col("edit_distance") > 0).filter(col("edit_distance") < 8).count()
direct_hits = df_silver.filter(col("edit_distance") == 0).count()

print(f"📊 Quality Metrics:")
print(f"   - Direct Sanctions Hits: {direct_hits}")
print(f"   - Evasive Patterns Caught: {evasions_count}")

# --- TEST 3: PROBABILISTIC DISTRIBUTION ---
# This ensures our Risk Scores are ready for Monte Carlo
print("\n📈 Risk Score Distribution (Monte Carlo Readiness):")
df_silver.groupBy("preliminary_risk_score").count().orderBy("preliminary_risk_score").show()

# --- TEST 4: DELTA TABLE INTEGRITY (Time Travel) ---
# Ensure we can see the history of this table
from delta.tables import DeltaTable
deltaTable = DeltaTable.forPath(spark, "s3a://silver/transactions_refined")
print("📜 Table History (Audit Trail):")
deltaTable.history().select("version", "timestamp", "operation", "operationParameters").show()

🔍 Sentinel Silver Validation: Starting...
✅ PASS: PII Masking confirmed. Raw User IDs removed.
📊 Quality Metrics:
   - Direct Sanctions Hits: 9985
   - Evasive Patterns Caught: 5036

📈 Risk Score Distribution (Monte Carlo Readiness):
+----------------------+-------+
|preliminary_risk_score|  count|
+----------------------+-------+
|                  0.01|9785226|
|                   0.4| 199753|
|                  0.95|  15021|
+----------------------+-------+

📜 Table History (Audit Trail):
+-------+-------------------+---------+--------------------+
|version|          timestamp|operation| operationParameters|
+-------+-------------------+---------+--------------------+
|      2|2026-02-08 12:10:47|    WRITE|{mode -> Overwrit...|
|      1|2026-02-08 12:04:35|    WRITE|{mode -> Overwrit...|
|      0|2026-02-08 11:41:43|    WRITE|{mode -> Overwrit...|
+-------+-------------------+---------+--------------------+

